In [1]:

import torch
import torch.nn as nn
import numpy as np
import os
import pandas as pd
import random
from scipy.signal import butter, filtfilt
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
torch.manual_seed(SEED)

In [2]:
def two_stage_predict(person_model, target_model, x):
    person_pred = person_model.predict(x)
    target_pred = np.zeros(len(x), dtype=int)

    genuine_mask = (person_pred == 1)
    if genuine_mask.sum() > 0:
        target_pred[genuine_mask] = target_model.predict(x[genuine_mask])

    return person_pred, target_pred

def pipeline_score(y_true_person, y_true_target, person_pred, target_pred):
    from sklearn.metrics import classification_report

    print("=== PERSON ===")
    print(classification_report(y_true_person, person_pred))

    genuine_mask = (person_pred == 1)
    if genuine_mask.sum() > 0:
        print("=== TARGET (genuine tahmin edilenlerde) ===")
        print(classification_report(
            y_true_target[genuine_mask],
            target_pred[genuine_mask]
        ))

    y_true_combined = ((y_true_person == 1) & (y_true_target == 1)).astype(int)
    y_pred_combined = ((person_pred   == 1) & (target_pred   == 1)).astype(int)
    print("=== PIPELINE TOPLAM (person=1 VE target=1) ===")
    print(classification_report(y_true_combined, y_pred_combined))

In [ ]:
x_train = np.load("train_features.npy")
y_train_target = np.load("train_target.npy")
y_train_person = np.load("train_person.npy")

x_test = np.load("test_features.npy")
y_test_target = np.load("test_target.npy")
y_test_person =np.load("test_person.npy")

x_diff = np.load("diff_features.npy")
y_diff_target = np.load("diff_target.npy")
y_diff_person = np.load("diff_person.npy")



# ── CatBoost ──────────────────────────────
person_model = CatBoostClassifier(
    iterations=500, learning_rate=0.1,
    depth=6, #l2_leaf_reg=10,
    verbose=100, random_seed=SEED
)
target_model = CatBoostClassifier(
    iterations=5000, learning_rate=0.1,
    class_weights=[1, 5], #eger ise yaramazsa kapat
    depth=6, #l2_leaf_reg=10,
    verbose=100, random_seed=SEED
)

# Person modeli: tum train verisi
person_model.fit(x_train, y_train_person)

# Target modeli: SADECE genuine ornekler
genuine_mask = (y_train_person == 1)
target_model.fit(x_train[genuine_mask], y_train_target[genuine_mask])

# ── Gorulmus session tahmini ──────────────
print("\n=== GORULMUS SESSION ===")
person_pred, target_pred = two_stage_predict(person_model, target_model, x_test)
pipeline_score(y_test_person, y_test_target, person_pred, target_pred)

# ── Gorulmemis session tahmini ────────────
print("\n=== GORULMEMIS SESSION (8. session) ===")
person_pred2, target_pred2 = two_stage_predict(person_model, target_model, x_diff)
pipeline_score(y_diff_person, y_diff_target, person_pred2, target_pred2)

0:	learn: 0.3968995	total: 43.8ms	remaining: 21.9s
100:	learn: 0.0002461	total: 3s	remaining: 11.8s
200:	learn: 0.0002460	total: 5.56s	remaining: 8.27s
300:	learn: 0.0002460	total: 8.12s	remaining: 5.37s
400:	learn: 0.0002460	total: 10.7s	remaining: 2.65s
499:	learn: 0.0002460	total: 13.3s	remaining: 0us
0:	learn: 0.5478644	total: 24ms	remaining: 1m 59s
100:	learn: 0.0829384	total: 2.39s	remaining: 1m 56s
200:	learn: 0.0476196	total: 4.77s	remaining: 1m 53s
300:	learn: 0.0272537	total: 7.15s	remaining: 1m 51s
400:	learn: 0.0173780	total: 9.49s	remaining: 1m 48s
500:	learn: 0.0121108	total: 11.9s	remaining: 1m 46s
600:	learn: 0.0084799	total: 14.2s	remaining: 1m 43s
700:	learn: 0.0064079	total: 16.6s	remaining: 1m 41s
800:	learn: 0.0051545	total: 18.9s	remaining: 1m 39s
900:	learn: 0.0042643	total: 21.3s	remaining: 1m 36s
1000:	learn: 0.0038522	total: 23.7s	remaining: 1m 34s
1100:	learn: 0.0035146	total: 26.1s	remaining: 1m 32s
1200:	learn: 0.0031518	total: 28.4s	remaining: 1m 29s
1300:

In [7]:
# ── Egitim tahmini ────────────
print("\n=== Egitim verisi Tahmini (8. session) ===")
person_pred2, target_pred2 = two_stage_predict(person_model, target_model, x_train)
pipeline_score(y_train_person, y_train_target, person_pred2, target_pred2)


=== Egitim verisi Tahmini (8. session) ===
=== PERSON ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      7854
           1       1.00      1.00      1.00      7746

    accuracy                           1.00     15600
   macro avg       1.00      1.00      1.00     15600
weighted avg       1.00      1.00      1.00     15600

=== TARGET (genuine tahmin edilenlerde) ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      6455
           1       1.00      1.00      1.00      1291

    accuracy                           1.00      7746
   macro avg       1.00      1.00      1.00      7746
weighted avg       1.00      1.00      1.00      7746

=== PIPELINE TOPLAM (person=1 VE target=1) ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     14309
           1       1.00      1.00      1.00      1291

    accuracy                    